In [27]:
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx
from langchain_ibm import WatsonxEmbeddings

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser

from langchain_community.document_loaders import  WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

from dotenv import load_dotenv
import os
import re
import bs4
from pprint import pprint


In [3]:
!pip install beautifulsoup4


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: c:\source\ollama\.venv\Scripts\python.exe -m pip install --upgrade pip


In [17]:

load_dotenv()

apiKey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.environ["HF_TOKEN"]
COHERE_API_KEY = os.environ["COHERE_API_KEY"]

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apiKey}",
    project_id=f"{project_id}"
)

watson_llm = ChatWatsonx(
  model_id="ibm/granite-4-h-small",
  url=f"{watsonx_ai_url}",
  api_key = f"{apiKey}",
  project_id=f"{project_id}",
  max_tokens = 2000,
  params = {
    "temperature":0
  }
)

In [7]:
web_loader = WebBaseLoader(
    web_paths=['https://n.news.naver.com/article/214/0001502756?cds=news_media_pc&type=editn'],
    bs_kwargs=dict(parse_only=bs4.SoupStrainer("article", attrs={"id":"dic_area"}))
)
web_docs = web_loader.load()

print(f"총 페이지 수: {len(web_docs)}")
print(f"총 페이지 텍스트: {web_docs}")
print(f"총 페이지 텍스트: {web_docs[0].page_content[:200]}")
print(f"메타 데이터: {web_docs[0].metadata}")

총 페이지 수: 1
총 페이지 텍스트: [Document(metadata={'source': 'https://n.news.naver.com/article/214/0001502756?cds=news_media_pc&type=editn'}, page_content='\n\n\n\n\n정부 출범 1주년을 맞은 이재명 대통령이 남은 임기 4년 동안 국정 속도를 두 배로 높여 국민의 삶에 더 큰 변화를 만들어내겠다고 다짐했습니다. 이 대통령은 오늘 제24회 국무회의에서 "앞으로 4년 동안 국정 속도를 두 배로 높이고 정성을 다하면 남은 시간은 8년과 같이 쓸 수 있다"며 "우리 국민의 삶과 대한민국에 더 큰 변화를 만들어내겠다"고 강조했습니다. 특히 "수출 등 핵심 지표 개선의 성과를 중소기업과 소상공인, 서민, 취약계층 등 민생 전반으로 확산시키는 데 주력해야 한다"며 "반도체뿐 아니라 로봇과 방산 등 여타 첨단 산업 육성에 박차를 가해 글로벌 초격차 경제 강국의 문을 활짝 열어가겠다"고 말했습니다. 지난 1년에 대해서는 "국민의 성원과 공직자의 헌신에 힘입어 내란에 따른 충격과 민생경제 혼란 등 위기를 잘 넘어왔다"며 "모두가 국민 여러분의 관심과 참여, 협력 덕분"이라고 공을 돌렸습니다. 그러면서 "임기를 시작할 때보다 마칠 때 더 많은 국민의 성원과 평가를 받는 정부가 되겠다는 말씀을 다시 한번 새긴다"며 "언제나 최선을 다하겠다"고 거듭 약속했습니다.\n')]
총 페이지 텍스트: 




정부 출범 1주년을 맞은 이재명 대통령이 남은 임기 4년 동안 국정 속도를 두 배로 높여 국민의 삶에 더 큰 변화를 만들어내겠다고 다짐했습니다. 이 대통령은 오늘 제24회 국무회의에서 "앞으로 4년 동안 국정 속도를 두 배로 높이고 정성을 다하면 남은 시간은 8년과 같이 쓸 수 있다"며 "우리 국민의 삶과 대한민국에 더 큰 변화를 만들어내겠다"고 
메타 데이터: {'source': 'https://n.news.naver.com/article/214/0001502756?cds=

In [8]:
pprint(web_docs[0].page_content)
pprint(web_docs[0].metadata)

('\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '정부 출범 1주년을 맞은 이재명 대통령이 남은 임기 4년 동안 국정 속도를 두 배로 높여 국민의 삶에 더 큰 변화를 만들어내겠다고 '
 '다짐했습니다. 이 대통령은 오늘 제24회 국무회의에서 "앞으로 4년 동안 국정 속도를 두 배로 높이고 정성을 다하면 남은 시간은 8년과 '
 '같이 쓸 수 있다"며 "우리 국민의 삶과 대한민국에 더 큰 변화를 만들어내겠다"고 강조했습니다. 특히 "수출 등 핵심 지표 개선의 성과를 '
 '중소기업과 소상공인, 서민, 취약계층 등 민생 전반으로 확산시키는 데 주력해야 한다"며 "반도체뿐 아니라 로봇과 방산 등 여타 첨단 산업 '
 '육성에 박차를 가해 글로벌 초격차 경제 강국의 문을 활짝 열어가겠다"고 말했습니다. 지난 1년에 대해서는 "국민의 성원과 공직자의 헌신에 '
 '힘입어 내란에 따른 충격과 민생경제 혼란 등 위기를 잘 넘어왔다"며 "모두가 국민 여러분의 관심과 참여, 협력 덕분"이라고 공을 '
 '돌렸습니다. 그러면서 "임기를 시작할 때보다 마칠 때 더 많은 국민의 성원과 평가를 받는 정부가 되겠다는 말씀을 다시 한번 새긴다"며 '
 '"언제나 최선을 다하겠다"고 거듭 약속했습니다.\n')
{'source': 'https://n.news.naver.com/article/214/0001502756?cds=news_media_pc&type=editn'}


In [10]:
content = web_docs[0].page_content.strip()

content

'정부 출범 1주년을 맞은 이재명 대통령이 남은 임기 4년 동안 국정 속도를 두 배로 높여 국민의 삶에 더 큰 변화를 만들어내겠다고 다짐했습니다. 이 대통령은 오늘 제24회 국무회의에서 "앞으로 4년 동안 국정 속도를 두 배로 높이고 정성을 다하면 남은 시간은 8년과 같이 쓸 수 있다"며 "우리 국민의 삶과 대한민국에 더 큰 변화를 만들어내겠다"고 강조했습니다. 특히 "수출 등 핵심 지표 개선의 성과를 중소기업과 소상공인, 서민, 취약계층 등 민생 전반으로 확산시키는 데 주력해야 한다"며 "반도체뿐 아니라 로봇과 방산 등 여타 첨단 산업 육성에 박차를 가해 글로벌 초격차 경제 강국의 문을 활짝 열어가겠다"고 말했습니다. 지난 1년에 대해서는 "국민의 성원과 공직자의 헌신에 힘입어 내란에 따른 충격과 민생경제 혼란 등 위기를 잘 넘어왔다"며 "모두가 국민 여러분의 관심과 참여, 협력 덕분"이라고 공을 돌렸습니다. 그러면서 "임기를 시작할 때보다 마칠 때 더 많은 국민의 성원과 평가를 받는 정부가 되겠다는 말씀을 다시 한번 새긴다"며 "언제나 최선을 다하겠다"고 거듭 약속했습니다.'

In [ ]:
lines = [line for line in content.split("\n") if line.strip()]
title = lines[0].split(".")

web_docs[0].metadata['title'] = title

['정부 출범 1주년을 맞은 이재명 대통령이 남은 임기 4년 동안 국정 속도를 두 배로 높여 국민의 삶에 더 큰 변화를 만들어내겠다고 다짐했습니다',
 ' 이 대통령은 오늘 제24회 국무회의에서 "앞으로 4년 동안 국정 속도를 두 배로 높이고 정성을 다하면 남은 시간은 8년과 같이 쓸 수 있다"며 "우리 국민의 삶과 대한민국에 더 큰 변화를 만들어내겠다"고 강조했습니다',
 ' 특히 "수출 등 핵심 지표 개선의 성과를 중소기업과 소상공인, 서민, 취약계층 등 민생 전반으로 확산시키는 데 주력해야 한다"며 "반도체뿐 아니라 로봇과 방산 등 여타 첨단 산업 육성에 박차를 가해 글로벌 초격차 경제 강국의 문을 활짝 열어가겠다"고 말했습니다',
 ' 지난 1년에 대해서는 "국민의 성원과 공직자의 헌신에 힘입어 내란에 따른 충격과 민생경제 혼란 등 위기를 잘 넘어왔다"며 "모두가 국민 여러분의 관심과 참여, 협력 덕분"이라고 공을 돌렸습니다',
 ' 그러면서 "임기를 시작할 때보다 마칠 때 더 많은 국민의 성원과 평가를 받는 정부가 되겠다는 말씀을 다시 한번 새긴다"며 "언제나 최선을 다하겠다"고 거듭 약속했습니다',
 '']

In [24]:
# 분할
chunk_size=300
chunk_overlap=30
splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
chunks = splitter.split_documents(web_docs)

In [25]:
# 백터스토어
vectorstore = FAISS.from_documents(documents=chunks, embedding=watson_embedding)
# vectorstore.similarity_search("이재명")

retriever = vectorstore.as_retriever(k=3)
rag_prompt = ChatPromptTemplate.from_template(
    """\
        당신은 뉴스 기사 qa 시스템 입니다.
        규칙:
        1. 제공된 기사 내용만 사용
        2. 기사 내용에 없는 정보는 추측하지 말고 '기사에서 확인할 수 없음' 이라 표시

        뉴스기사:
        {context}

        질문:
        {question}

        답변: 
    """
)


In [ ]:
# 체인 생성 질의

def format_docs(docs):
    """Document 객체에서 page_content 추출"""
    return "\n\n".join([d.page_content for d in docs])

retriever = vectorstore.as_retriever(search_kwargs={'k':5})

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 컨텍스트를 참고하여 질문에 답하세요\n 컨텍스트에 없는 내용은 모른다 라고 답하세요\n\n컨텍스트:\n{context}"),
    ("human", "{question}")
])

chain = {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        } | rag_prompt | watson_llm | StrOutputParser()


response = chain.invoke("기사가 뭔내용이여?")


In [30]:
print(response)

이 기사는 대한민국의 대통령 이재명이 정부 출범 1주년을 맞아 남은 임기 4년 동안 국정 속도를 두 배로 높여 국민의 삶에 더 큰 변화를 만들어내겠다고 다짐한 내용을 담고 있습니다. 이 대통령은 지난 1년간 국민의 성원과 공직자의 헌신에 힘입어 위기를 잘 넘어왔으며, 앞으로는 반도체뿐 아니라 로봇과 방산 등 여타 첨단 산업 육성에 박차를 가해 글로벌 초격차 경제 강국의 문을 활짝 열어가겠다고 말했습니다.
